In [1]:
!pip install pandas kafka-python pyspark==3.5.0 findspark tensorflow

  Obtaining dependency information for numpy>=1.23.2 from https://files.pythonhosted.org/packages/c1/ca/2f384720020c7b244d22508cb7ab23d95f179fcfff33c31a6eeba8d6c512/numpy-2.0.2-cp311-cp311-macosx_14_0_arm64.whl.metadata
  Using cached numpy-2.0.2-cp311-cp311-macosx_14_0_arm64.whl.metadata (60 kB)
Using cached numpy-2.0.2-cp311-cp311-macosx_14_0_arm64.whl (5.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.2
    Uninstalling numpy-2.3.2:
      Successfully uninstalled numpy-2.3.2

[notice] A new release of pip is available: 23.2.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import findspark
from pyspark.sql import SparkSession
import os
import pickle
import tensorflow as tf
import numpy as np
import pandas as pd
from pyspark.sql.types import StructType, IntegerType, StructField, StringType, TimestampType, DoubleType
from pyspark.sql.functions import col, from_json, current_timestamp, hour, dayofweek, to_timestamp, pandas_udf
from tensorflow.keras.models import load_model

In [3]:
findspark.init()

# Cấu hình Spark với Kafka
scala_version = '2.12'
spark_version = '3.5.0'

packages = [
    f'org.apache.spark:spark-sql-kafka-0-10_{scala_version}:{spark_version}',
    'org.apache.kafka:kafka-clients:3.5.0'
]

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("traffic-lstm-prediction") \
    .config("spark.jars.packages", ",".join(packages)) \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark Version: {spark.version}")

25/09/29 22:29:44 WARN Utils: Your hostname, 193499-HHVTien.local resolves to a loopback address: 127.0.0.1; using 192.168.2.250 instead (on interface en0)
25/09/29 22:29:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/hotien/.ivy2/cache
The jars for the packages stored in: /Users/hotien/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
org.apache.kafka#kafka-clients added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-bfc2db83-944c-4562-bebd-d97a21234ff0;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
	found org.apache.kafka#kafka-clients;3.5.0 in central
	found com.github.luben#zstd-jni;1.5.5-1 in central
	found org.lz4#lz4-java;1.8.0 in central
:: resolution report :: resolve 228ms :: artifacts dl 12ms
	:: modules in use:
	com.github.luben#zstd-jni;1.5.5-1 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from central in [default]
	org.apache.commons#commons-pool2;2.11.1 f

Spark Version: 3.5.0


In [4]:
# Schema cho dữ liệu từ Kafka
vehicle_details_schema = StructType([
    StructField("car", IntegerType(), True),
    StructField("motorbike", IntegerType(), True),
    StructField("bus", IntegerType(), True),
    StructField("truck", IntegerType(), True)
])

kafka_schema = StructType([
    StructField("timestamp_utc", StringType(), True),
    StructField("location", StringType(), True),
    StructField("total_vehicles", IntegerType(), True),
    StructField("vehicle_details", vehicle_details_schema, True),
    StructField("traffic_status", IntegerType(), True)
])

topic_name = 'traffic_detection_topic'
kafka_server = 'localhost:9092'

In [5]:
# Đọc stream từ Kafka
kafka_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_server) \
    .option("subscribe", topic_name) \
    .option("startingOffsets", "latest") \
    .load()

# Parse dữ liệu JSON
parsed_df = kafka_df.selectExpr("CAST(value AS STRING) as json") \
    .select(from_json(col("json"), kafka_schema).alias("data")) \
    .select("data.*")

# Chuyển đổi timestamp và thêm features
final_df = parsed_df \
    .withColumn("timestamp", to_timestamp(col("timestamp_utc"))) \
    .withColumn("hour", hour(col("timestamp"))) \
    .withColumn("dayofweek", dayofweek(col("timestamp"))) \
    .withColumn("processing_time", current_timestamp())

print("Schema của dữ liệu đã parse:")
final_df.printSchema()

Schema của dữ liệu đã parse:
root
 |-- timestamp_utc: string (nullable = true)
 |-- location: string (nullable = true)
 |-- total_vehicles: integer (nullable = true)
 |-- vehicle_details: struct (nullable = true)
 |    |-- car: integer (nullable = true)
 |    |-- motorbike: integer (nullable = true)
 |    |-- bus: integer (nullable = true)
 |    |-- truck: integer (nullable = true)
 |-- traffic_status: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- hour: integer (nullable = true)
 |-- dayofweek: integer (nullable = true)
 |-- processing_time: timestamp (nullable = false)



In [6]:
# Đường dẫn model
model_dir = "./models"
model_path = os.path.join(model_dir, "traffic_lstm_model.h5")
feature_columns_path = os.path.join(model_dir, "traffic_features.pkl")
scaler_path = os.path.join(model_dir, "traffic_scaler.pkl")
label_encoder_path = os.path.join(model_dir, "traffic_label_encoder.pkl")

os.makedirs(model_dir, exist_ok=True)

# Tải các thành phần đã lưu
try:
    model = load_model(model_path)
    with open(feature_columns_path, 'rb') as f:
        feature_columns = pickle.load(f)
    with open(scaler_path, 'rb') as f:
        scaler = pickle.load(f)
    with open(label_encoder_path, 'rb') as f:
        label_encoder = pickle.load(f)

    print("Đã tải thành công model và các thành phần")

except Exception as e:
    print(f"Lỗi khi tải model: {e}")
    print("Tạo model mẫu để test...")

    # Tạo model mẫu cho testing
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense
    from sklearn.preprocessing import StandardScaler, LabelEncoder

    # Model đơn giản
    model = Sequential([
        LSTM(50, activation='relu', input_shape=(1, 6)),
        Dense(25, activation='relu'),
        Dense(5, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Các thành phần khác
    feature_columns = ['car', 'motorbike', 'bus', 'truck', 'hour', 'dayofweek']
    scaler = StandardScaler()
    label_encoder = LabelEncoder()
    label_encoder.fit(['RẤT_THƯA', 'THƯA', 'BÌNH_THƯỜNG', 'ĐÔNG', 'RẤT_ĐÔNG'])

# Broadcast các thành phần
feature_columns_bc = spark.sparkContext.broadcast(feature_columns)
scaler_bc = spark.sparkContext.broadcast(scaler)
model_bc = spark.sparkContext.broadcast(model)
label_encoder_bc = spark.sparkContext.broadcast(label_encoder)

/Users/hotien/Documents/cao học/HK5/chuyên đề dự đoán/pj/traffic-forecast/.venv/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.0 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/hotien/Documents/cao học/HK5/chuyên đề dự đoán/pj/traffic-forecast/.venv/lib/python3.11/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.7.0 when using version 1.7.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Đã tải thành công model và các thành phần


In [7]:
# Sửa UDF - đảm bảo trả về đúng kiểu String
@pandas_udf(StringType())
def predict_traffic_udf(car: pd.Series, motorbike: pd.Series, bus: pd.Series,
                        truck: pd.Series, hour: pd.Series, dayofweek: pd.Series) -> pd.Series:
    try:
        # Tạo DataFrame từ các features
        features_df = pd.DataFrame({
            "car": car,
            "motorbike": motorbike,
            "bus": bus,
            "truck": truck,
            "hour": hour,
            "dayofweek": dayofweek
        })

        # Lấy các thành phần từ broadcast
        feature_columns = feature_columns_bc.value
        scaler = scaler_bc.value
        model = model_bc.value
        label_encoder = label_encoder_bc.value

        # Chuẩn hóa dữ liệu
        features_scaled = scaler.transform(features_df[feature_columns])

        # Reshape cho LSTM (batch_size, sequence_length=1, num_features)
        features_reshaped = features_scaled.reshape(-1, 1, len(feature_columns))

        # Dự đoán
        predictions = model.predict(features_reshaped, verbose=0)
        predicted_classes = np.argmax(predictions, axis=1)

        # Chuyển đổi lại nhãn gốc - đảm bảo là string
        predicted_labels = label_encoder.inverse_transform(predicted_classes)

        # Convert to string explicitly
        predicted_labels = [str(x) for x in predicted_labels]

        return pd.Series(predicted_labels)

    except Exception as e:
        print(f"Lỗi trong dự đoán: {e}")
        # Trả về giá trị mặc định dạng string
        return pd.Series(["LỖI"] * len(car))

In [8]:
# Áp dụng UDF cho dự đoán
df_pred = final_df.withColumn(
    "predicted_traffic_status",
    predict_traffic_udf(
        col("vehicle_details.car"),
        col("vehicle_details.motorbike"),
        col("vehicle_details.bus"),
        col("vehicle_details.truck"),
        col("hour"),
        col("dayofweek")
    )
)

# Chọn các cột để hiển thị
output_df = df_pred.select(
    "timestamp",
    "location",
    "vehicle_details",
    "traffic_status",
    "predicted_traffic_status",
    "processing_time"
)

In [9]:
# Ghi kết quả ra console
query = output_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .option("truncate", "false") \
    .trigger(processingTime='30 seconds') \
    .start()

print("Bắt đầu xử lý streaming...")
query.awaitTermination()

25/09/29 22:30:11 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /private/var/folders/yp/zzs1vcsn3_j60vflhd6qr_m40000gn/T/temporary-3f16ba5e-1d66-42ae-8816-7fbc54e69a62. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.


Bắt đầu xử lý streaming...
-------------------------------------------
Batch: 0
-------------------------------------------
+---------+--------+---------------+--------------+------------------------+---------------+
|timestamp|location|vehicle_details|traffic_status|predicted_traffic_status|processing_time|
+---------+--------+---------------+--------------+------------------------+---------------+
+---------+--------+---------------+--------------+------------------------+---------------+



25/09/29 22:30:30 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
/Users/hotien/Documents/cao học/HK5/chuyên đề dự đoán/pj/traffic-forecast/.venv/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 22 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------------+---------------+---------------+--------------+------------------------+-----------------------+
|timestamp                 |location       |vehicle_details|traffic_status|predicted_traffic_status|processing_time        |
+--------------------------+---------------+---------------+--------------+------------------------+-----------------------+
|2025-09-29 15:30:21.124617|CMT8_PhamVanHai|{1, 0, 0, 1}   |1             |1                       |2025-09-29 22:30:30.021|
+--------------------------+---------------+---------------+--------------+------------------------+-----------------------+



25/09/29 22:31:00 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894
25/09/29 22:31:00 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894


...

-------------------------------------------
Batch: 152
-------------------------------------------
+--------------------------+---------------+---------------+--------------+------------------------+-----------------------+
|timestamp                 |location       |vehicle_details|traffic_status|predicted_traffic_status|processing_time        |
+--------------------------+---------------+---------------+--------------+------------------------+-----------------------+
|2025-09-29 16:45:38.995963|CMT8_PhamVanHai|{0, 0, 0, 0}   |1             |1                       |2025-09-29 23:46:00.012|
|2025-09-29 16:45:54.615837|CMT8_PhamVanHai|{3, 1, 0, 0}   |1             |1                       |2025-09-29 23:46:00.012|
+--------------------------+---------------+---------------+--------------+------------------------+-----------------------+



25/09/30 00:02:12 WARN KafkaDataConsumer: KafkaDataConsumer is not running in UninterruptibleThread. It may hang when KafkaDataConsumer's methods are interrupted because of KAFKA-1894


-------------------------------------------
Batch: 153
-------------------------------------------
+--------------------------+---------------+---------------+--------------+------------------------+-----------------------+
|timestamp                 |location       |vehicle_details|traffic_status|predicted_traffic_status|processing_time        |
+--------------------------+---------------+---------------+--------------+------------------------+-----------------------+
|2025-09-29 16:46:10.215466|CMT8_PhamVanHai|{1, 0, 0, 1}   |1             |1                       |2025-09-30 00:02:12.472|
+--------------------------+---------------+---------------+--------------+------------------------+-----------------------+



25/09/30 00:52:00 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 922541 ms exceeds timeout 120000 ms
25/09/30 00:52:00 WARN SparkContext: Killing executors is not supported by current scheduler.
25/09/30 01:09:26 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

Py4JError: An error occurred while calling o144.awaitTermination